> **Public release note.**.  Notebook outputs have been removed because the underlying Malaysian Motor claims data are confidential. Execution counts have also been cleared to provide clean public versions of the notebooks. Local user-specific paths and individual claim identifiers have been removed. The repository documents the data-processing, modelling and validation workflow used in the dissertation, but the numerical results cannot be reproduced end-to-end without the confidential input data.


# 05.7A — Clean Random Forest Future-Development Model

**Purpose:** rerun the Random Forest future-development model using only the
predictors saved by `05_4A_Clean_Snapshot_Model_Inputs.ipynb`.

This notebook:

- cannot accidentally use targets, split labels or later valuation fields;
- fits the same transparent baseline specification at DEV_QTR 4, 8 and 12;
- evaluates the future-development target and reconstructed latest incurred amount;
- saves test predictions, accident-year summaries and fitted models;
- exports a clean worked-claim record where available.

Previous Random Forest outputs should be treated as superseded until this
notebook has been run successfully.

This part tells the computer exactly which folders to retrieve data from, creates a new folder to save the final results, and reads a simple list of instructions specifying which specific information to use to predict future claims.

In [ ]:
from pathlib import Path
import json
import warnings

import joblib
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

PROJECT_FOLDER = Path(
    "/path/to/BI_large_claims_project"
)

INPUT_FOLDER = (
    PROJECT_FOLDER
    / "processed"
    / "chapter5_outputs"
    / "section_5_4A_clean_snapshot_model_inputs"
)

OUTPUT_FOLDER = (
    PROJECT_FOLDER
    / "processed"
    / "chapter5_outputs"
    / "section_5_7A_clean_rf_future_development"
)
OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)

with open(INPUT_FOLDER / "feature_manifest.json") as f:
    manifest = json.load(f)

SNAPSHOTS = manifest["snapshots"]
FEATURE_COLUMNS = manifest["feature_columns"]
CATEGORICAL_FEATURES = manifest["categorical_features"]
NUMERIC_FEATURES = manifest["numeric_features"]
TARGET_LATEST = manifest["target_latest"]
TARGET_FUTURE = manifest["target_future"]
SNAPSHOT_EXCESS = manifest["snapshot_excess"]

print("Input folder exists:", INPUT_FOLDER.exists())
print("Output folder:", OUTPUT_FOLDER)
print("Feature count:", len(FEATURE_COLUMNS))

## Main Random Forest specification

The initial clean rerun deliberately keeps the earlier model settings so that
any change in results can be attributed to removing leakage and correcting the
feature pipeline.

A separate calibration notebook can then vary `min_samples_leaf` and
`max_features` once the corrected baseline is established.

The code sets out the particular rules and limits concerning the way in which the RF model should analyse the data, for example by restricting its complexity and by requiring that each final group contains at least 50 claims. It then stores these precise settings in a reference file so that anyone looking at the project at a later stage can see exactly how the model was configured to produce its results.

In [ ]:
RF_PARAMS = {
    "n_estimators": 200,
    "max_depth": 8,
    "min_samples_leaf": 50,
    "min_samples_split": 2,
    "max_features": 1.0,
    "bootstrap": True,
    "criterion": "squared_error",
    "random_state": 42,
    "n_jobs": -1,
}

with open(OUTPUT_FOLDER / "rf_main_parameters.json", "w") as f:
    json.dump(RF_PARAMS, f, indent=2)

print(RF_PARAMS)

This creates an automatic process for getting the data ready and checking how well the model works. It defines how to handle missing information; for example, by replacing empty number values with the middle value and empty text values with the most common category before sending the data to the Random Forest algorithm. It also includes some math helper functions to check the model's accuracy (including the WAPE metric) and to make sure the program does not stop working if a division by zero happens.

In [ ]:
def make_one_hot_encoder():
    # Compatible with both newer and older scikit-learn releases.
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=True)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=True)


def build_model():
    numeric_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
        ]
    )

    categorical_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", make_one_hot_encoder()),
        ]
    )

    preprocessor = ColumnTransformer(
        transformers=[
            ("numeric", numeric_transformer, NUMERIC_FEATURES),
            ("categorical", categorical_transformer, CATEGORICAL_FEATURES),
        ],
        remainder="drop",
    )

    model = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("regressor", RandomForestRegressor(**RF_PARAMS)),
        ]
    )
    return model


def safe_ratio(numerator, denominator):
    if denominator == 0:
        return np.nan
    return numerator / denominator


def wape(y_true, y_pred):
    denominator = np.abs(np.asarray(y_true)).sum()
    if denominator == 0:
        return np.nan
    return np.abs(np.asarray(y_true) - np.asarray(y_pred)).sum() / denominator

This function loads the saved data at the current claim's progress, splitting it into the inputs the model will use and the final results it will predict. Before sending the data on, it performs careful quality checks to ensure all rows match correctly and that no variables are included by mistake. This ensures that the model learns only from the exact, clean information set up at the start, without any data errors.

In [ ]:
def load_snapshot(snapshot):
    folder = INPUT_FOLDER / snapshot

    X_train = pd.read_parquet(folder / "X_train.parquet")
    X_test = pd.read_parquet(folder / "X_test.parquet")
    targets_train = pd.read_parquet(folder / "targets_train.parquet")
    targets_test = pd.read_parquet(folder / "targets_test.parquet")
    meta_train = pd.read_parquet(folder / "meta_train.parquet")
    meta_test = pd.read_parquet(folder / "meta_test.parquet")

    if list(X_train.columns) != FEATURE_COLUMNS:
        raise ValueError(f"{snapshot}: X_train columns differ from manifest.")
    if list(X_test.columns) != FEATURE_COLUMNS:
        raise ValueError(f"{snapshot}: X_test columns differ from manifest.")

    if len(X_train) != len(targets_train) or len(X_train) != len(meta_train):
        raise ValueError(f"{snapshot}: train files are not row-aligned.")
    if len(X_test) != len(targets_test) or len(X_test) != len(meta_test):
        raise ValueError(f"{snapshot}: test files are not row-aligned.")

    forbidden_present = set(manifest["forbidden_features"]).intersection(X_train.columns)
    if forbidden_present:
        raise ValueError(
            f"{snapshot}: forbidden predictors found: {sorted(forbidden_present)}"
        )

    return X_train, X_test, targets_train, targets_test, meta_train, meta_test

## Fit, validate and save

The model is fitted to future incurred BI Excess development.

Two prediction forms are retained:

- **raw future development**, which may be positive or negative;
- **deployed projected latest amount**, where the reconstructed BI Excess
  amount is floored at zero.

The raw target is used for claim-level MAE and RMSE. The reconstructed latest
amount is used for reserve calibration and accident-year comparisons.

This is the main loop for Random Forest modeling. For each development snapshot, it loads the training and test data, fits the Random Forest model to predict future BI Excess incurred development, and then adds that prediction to the amount already incurred at the snapshot to get a projected later BI Excess amount.

Next, it measures performance at both the claim and accident-year levels, using metrics like MAE, RMSE, bias ratio, and WAPE. The process saves the predictions, accident-year summaries, and the fitted model. It also creates a full-snapshot diagnostic that shows training and test results separately, but leaves out individual worked-claim records in the public version.

In [ ]:
overall_results = []
ay_results_list = []
full_snapshot_results = []
worked_claim_rows = []  # kept for compatibility; record-level export omitted publicly

for snapshot in SNAPSHOTS:
    print("\n" + "=" * 72)
    print("Fitting", snapshot)

    (
        X_train,
        X_test,
        targets_train,
        targets_test,
        meta_train,
        meta_test,
    ) = load_snapshot(snapshot)

    y_train_future = targets_train[TARGET_FUTURE].astype(float)
    y_test_future = targets_test[TARGET_FUTURE].astype(float)

    actual_latest_test = targets_test[TARGET_LATEST].astype(float).to_numpy()
    snapshot_excess_test = targets_test[SNAPSHOT_EXCESS].astype(float).to_numpy()

    model = build_model()
    model.fit(X_train, y_train_future)

    predicted_future_raw = model.predict(X_test)
    predicted_latest_raw = snapshot_excess_test + predicted_future_raw
    predicted_latest_deployed = np.maximum(predicted_latest_raw, 0.0)
    predicted_future_deployed = (
        predicted_latest_deployed - snapshot_excess_test
    )

    claim_mae_future = mean_absolute_error(
        y_test_future, predicted_future_raw
    )
    claim_rmse_future = np.sqrt(
        mean_squared_error(y_test_future, predicted_future_raw)
    )

    actual_latest_total = float(actual_latest_test.sum())
    snapshot_total = float(snapshot_excess_test.sum())
    actual_future_total = float(y_test_future.sum())
    predicted_future_total_raw = float(predicted_future_raw.sum())
    predicted_future_total_deployed = float(predicted_future_deployed.sum())
    predicted_latest_total = float(predicted_latest_deployed.sum())

    prediction_df = meta_test.copy()
    prediction_df["snapshot"] = snapshot
    prediction_df["actual_snapshot_bixs"] = snapshot_excess_test
    prediction_df["actual_future_development"] = y_test_future.to_numpy()
    prediction_df["actual_latest_bixs"] = actual_latest_test
    prediction_df["predicted_future_development_raw"] = predicted_future_raw
    prediction_df["predicted_latest_bixs_raw"] = predicted_latest_raw
    prediction_df["predicted_latest_bixs"] = predicted_latest_deployed
    prediction_df["predicted_future_development_deployed"] = (
        predicted_future_deployed
    )
    prediction_df["latest_error"] = (
        prediction_df["predicted_latest_bixs"]
        - prediction_df["actual_latest_bixs"]
    )
    prediction_df["latest_absolute_error"] = prediction_df["latest_error"].abs()

    overall_results.append({
        "model": "Random Forest",
        "snapshot": snapshot,
        "test_claims": len(X_test),
        "actual_snapshot_bixs_total": snapshot_total,
        "actual_future_development_total": actual_future_total,
        "predicted_future_development_total_raw": predicted_future_total_raw,
        "predicted_future_development_total_deployed": predicted_future_total_deployed,
        "actual_latest_bixs_total": actual_latest_total,
        "predicted_latest_bixs_total": predicted_latest_total,
        "latest_difference_predicted_minus_actual": (
            predicted_latest_total - actual_latest_total
        ),
        "latest_bias_ratio": safe_ratio(
            predicted_latest_total, actual_latest_total
        ),
        "claim_level_future_mae": claim_mae_future,
        "claim_level_future_rmse": claim_rmse_future,
        "claim_level_latest_wape": wape(
            actual_latest_test, predicted_latest_deployed
        ),
    })

    if "ACC_YEAR" not in prediction_df.columns:
        raise ValueError(f"{snapshot}: ACC_YEAR missing from metadata.")

    ay_summary = (
        prediction_df
        .groupby("ACC_YEAR", as_index=False)
        .agg(
            claim_count=("CLAIMS_KEY", "count"),
            actual_snapshot_bixs=("actual_snapshot_bixs", "sum"),
            actual_future_development=("actual_future_development", "sum"),
            predicted_future_development_raw=(
                "predicted_future_development_raw", "sum"
            ),
            predicted_future_development_deployed=(
                "predicted_future_development_deployed", "sum"
            ),
            actual_latest_bixs=("actual_latest_bixs", "sum"),
            predicted_latest_bixs=("predicted_latest_bixs", "sum"),
        )
    )

    ay_summary["model"] = "Random Forest"
    ay_summary["snapshot"] = snapshot
    ay_summary["latest_error"] = (
        ay_summary["predicted_latest_bixs"]
        - ay_summary["actual_latest_bixs"]
    )
    ay_summary["latest_absolute_error"] = ay_summary["latest_error"].abs()
    ay_summary["latest_percentage_error"] = np.where(
        ay_summary["actual_latest_bixs"] != 0,
        ay_summary["latest_error"] / ay_summary["actual_latest_bixs"],
        np.nan,
    )
    ay_summary["latest_bias_ratio"] = np.where(
        ay_summary["actual_latest_bixs"] != 0,
        ay_summary["predicted_latest_bixs"]
        / ay_summary["actual_latest_bixs"],
        np.nan,
    )

    ay_wape = safe_ratio(
        ay_summary["latest_absolute_error"].sum(),
        ay_summary["actual_latest_bixs"].abs().sum(),
    )
    overall_results[-1]["accident_year_latest_wape"] = ay_wape

    snapshot_output = OUTPUT_FOLDER / snapshot
    snapshot_output.mkdir(parents=True, exist_ok=True)

    prediction_df.to_parquet(
        snapshot_output / "rf_test_predictions.parquet",
        index=False,
    )
    ay_summary.to_csv(
        snapshot_output / "rf_test_predictions_by_accident_year.csv",
        index=False,
    )
    joblib.dump(
        model,
        snapshot_output / f"rf_future_development_{snapshot}.joblib",
    )

    ay_results_list.append(ay_summary)

    # Full-snapshot diagnostic: training rows are in-sample and are labelled as such.
    predicted_future_train = model.predict(X_train)
    snapshot_excess_train = (
        targets_train[SNAPSHOT_EXCESS].astype(float).to_numpy()
    )
    predicted_latest_train = np.maximum(
        snapshot_excess_train + predicted_future_train,
        0.0,
    )

    train_predictions = meta_train.copy()
    train_predictions["validation_status"] = "training_in_sample"
    train_predictions["actual_snapshot_bixs"] = snapshot_excess_train
    train_predictions["actual_latest_bixs"] = (
        targets_train[TARGET_LATEST].astype(float).to_numpy()
    )
    train_predictions["predicted_future_development_raw"] = (
        predicted_future_train
    )
    train_predictions["predicted_latest_bixs"] = predicted_latest_train

    test_full = prediction_df[
        list(meta_test.columns)
        + [
            "actual_snapshot_bixs",
            "actual_latest_bixs",
            "predicted_future_development_raw",
            "predicted_latest_bixs",
        ]
    ].copy()
    test_full["validation_status"] = "held_out_test"

    full_predictions = pd.concat(
        [train_predictions, test_full],
        ignore_index=True,
        sort=False,
    )
    full_predictions["snapshot"] = snapshot
    full_predictions.to_parquet(
        snapshot_output / "rf_full_snapshot_diagnostic_predictions.parquet",
        index=False,
    )

    full_summary = (
        full_predictions
        .groupby(["snapshot", "validation_status"], as_index=False)
        .agg(
            claim_count=("CLAIMS_KEY", "count"),
            actual_snapshot_bixs=("actual_snapshot_bixs", "sum"),
            actual_latest_bixs=("actual_latest_bixs", "sum"),
            predicted_latest_bixs=("predicted_latest_bixs", "sum"),
        )
    )
    full_snapshot_results.append(full_summary)

    # Record-level worked-claim export is omitted from the public release.

    print(
        snapshot,
        "actual latest=", f"{actual_latest_total:,.2f}",
        "predicted latest=", f"{predicted_latest_total:,.2f}",
        "bias=", f"{safe_ratio(predicted_latest_total, actual_latest_total):.3f}",
        "AY WAPE=", f"{ay_wape:.3f}",
    )

This is the reporting/export step that packages the Random Forest results into the tables used for later comparison and dissertation analysis.

In [ ]:
overall_results_df = pd.DataFrame(overall_results)
ay_results_df = pd.concat(ay_results_list, ignore_index=True)
full_snapshot_results_df = pd.concat(
    full_snapshot_results,
    ignore_index=True,
)

overall_results_df.to_csv(
    OUTPUT_FOLDER / "rf_clean_test_summary_by_snapshot.csv",
    index=False,
)
ay_results_df.to_csv(
    OUTPUT_FOLDER / "rf_clean_test_summary_by_snapshot_and_accident_year.csv",
    index=False,
)
full_snapshot_results_df.to_csv(
    OUTPUT_FOLDER / "rf_full_snapshot_diagnostic_summary.csv",
    index=False,
)

print("Record-level worked-claim export omitted from public release.")

display(overall_results_df)
print("Saved corrected RF outputs to:", OUTPUT_FOLDER)

summary table

In [ ]:
overall_results_df

setting up the individual claim example - restrcting to the claims with no BI Excess. 

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

snapshot = "DEV_QTR_4"

prediction_path = (
    OUTPUT_FOLDER
    / snapshot
    / "rf_test_predictions.parquet"
)

worked_predictions = pd.read_parquet(prediction_path)

worked_example_candidates = worked_predictions[
    (worked_predictions["actual_snapshot_bixs"] == 0)
    & (worked_predictions["actual_latest_bixs"] > 0)
    & (worked_predictions["predicted_latest_bixs"] > 0)
].copy()

worked_example_candidates["absolute_error"] = (
    worked_example_candidates["predicted_latest_bixs"]
    - worked_example_candidates["actual_latest_bixs"]
).abs()

print("Eligible held-out claims:", len(worked_example_candidates))

if worked_example_candidates.empty:
    raise ValueError("No eligible DEV_QTR_4 worked-example claims were found.")

median_absolute_error = worked_example_candidates["absolute_error"].median()

worked_example_candidates["distance_from_median_error"] = (
    worked_example_candidates["absolute_error"]
    - median_absolute_error
).abs()

worked_example = (
    worked_example_candidates
    .sort_values("distance_from_median_error")
    .head(1)
)

display(worked_example)